# Build the eddy-centre stratification cache

This notebook calculates environmental buoyancy frequency with `xroms` and writes one row per existing eddy-day. It **does not calculate or modify tilt**. The documented xroms sequence is `roms_dataset → density → N2`; depth-weighted values are summarized over the upper 200 and 500 m.

Run this once on Katana before notebook 02. Inspect the model variable names and the sign/range checks before accepting the cache.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


In [ ]:
MODEL_ROOT = Path("/srv/scratch/z3533156/26year_BRAN2020")
N2_CACHE = Path("/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/n2_eddy_day.parquet")

required_model_columns = ["Eddy", "Day", "ic", "jc"]
df[required_model_columns].isna().mean().rename("missing_fraction")


In [ ]:
# The full run is intentionally explicit because it opens every required ROMS file.
n2_cache = mech.build_n2_cache_xroms(
    df,
    MODEL_ROOT,
    N2_CACHE,
    depths=(200, 500),
)
n2_cache.head()


In [ ]:
display(n2_cache[["N2_200m_s2", "N2_500m_s2"]].describe(percentiles=[.01, .05, .5, .95, .99]))
assert not n2_cache.duplicated(["Eddy", "Day"]).any()
for col in ["N2_200m_s2", "N2_500m_s2"]:
    print(col, "positive fraction:", (n2_cache[col] > 0).mean(), "missing:", n2_cache[col].isna().mean())

n2_cache[["N2_200m_s2", "N2_500m_s2"]].plot.hist(bins=80, logy=True, alpha=.55)
plt.xlabel(r"$N^2$ (s$^{-2}$)")
plt.title("Eddy-centre depth-mean stratification QC")
plt.show()


## Acceptance checks

- Confirm `temp`, `salt`, ROMS vertical coordinates, `ic → xi_rho`, and `jc → eta_rho` on several manually selected profiles.
- Investigate negative or extreme values rather than silently clipping them. The depth means retain resolved negative values.
- Compare several cached columns against direct plots of `N2(z)`.
- Record the installed `xroms` version with the final results.


In [ ]:
import xroms
print("xroms", xroms.__version__)
print("cache", N2_CACHE)
